# 02 — Data Cleaning Pipeline
### Bluestock MF Capstone · Day 2

This notebook documents every cleaning step applied to all **10 source CSVs**.  
Outputs are written to `data/processed/` and loaded into `data/db/bluestock_mf.db`.

**Sections**
1. Setup & Imports  
2. Task 1 — `nav_history.csv`  
3. Task 2 — `investor_transactions.csv` (SIP + Lumpsum unified)  
4. Task 3 — `scheme_performance.csv` (returns + expense ratio)  
5. Tasks 4–10 — Remaining 7 source files  
6. Task 4 — SQLite Star Schema (DDL)  
7. Task 5 — Load into SQLite + Row-Count Verification  
8. Task 6 — 10 Analytical SQL Queries  
9. Task 7 — Data Dictionary Preview  
10. Task 8 — Git Commit  


---
## 0 · Setup & Imports

In [1]:
import sqlite3, subprocess
from pathlib import Path

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

BASE  = Path(".").resolve()
RAW   = BASE / "data" / "raw"
PROC  = BASE / "data" / "processed"
DB    = BASE / "data" / "db"
SQL   = BASE / "sql"

for d in (PROC, DB, SQL):
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DB / "bluestock_mf.db"

print("Working directory:", BASE)
print("Raw files found  :", [f.name for f in RAW.glob("*.csv")])


Working directory: C:\Users\abhin\Downloads\files\bluestock_mf_capstone_day1\bluestock_mf_capstone\notebooks
Raw files found  : []


### Raw-file audit — shape, nulls, duplicates

In [2]:
raw_files = sorted(RAW.glob("*.csv"))
rows = []
for f in raw_files:
    df = pd.read_csv(f)
    rows.append({
        "file":       f.name,
        "rows":       len(df),
        "cols":       df.shape[1],
        "null_cells": int(df.isna().sum().sum()),
        "dup_rows":   int(df.duplicated().sum()),
        "columns":    ", ".join(df.columns.tolist()[:5]) + (" …" if df.shape[1]>5 else ""),
    })

audit = pd.DataFrame(rows)
display(audit)


""


---
## Task 1 · Clean `nav_history.csv`

**Rules applied**
| # | Rule | Detail |
|---|---|---|
| 1 | Parse dates | `pd.to_datetime` → ISO-8601 string |
| 2 | Sort | by `amfi_code` then `nav_date` ascending |
| 3 | Remove duplicates | on `(amfi_code, nav_date)` |
| 4 | Forward-fill gaps | `resample('B').ffill()` per fund — fills weekends & Indian market holidays |
| 5 | Validate NAV > 0 | Drop rows where `nav ≤ 0` |


In [7]:
nav_raw = pd.read_csv( RAW/"02_nav_history.csv")
print(f"Source : {len(nav_raw):,} rows | {nav_raw.amfi_code.nunique()} funds")
print(f"Columns: {nav_raw.columns.tolist()}")
print(f"Sample :")
display(nav_raw.head(3))


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\abhin\\Downloads\\files\\bluestock_mf_capstone_day1\\bluestock_mf_capstone\\notebooks\\data\\raw\\02_nav_history.csv'

In [ ]:
# 1a · Parse dates
nav_raw["nav_date"] = pd.to_datetime(nav_raw["nav_date"], errors="coerce")
bad_dates = nav_raw["nav_date"].isna().sum()
print(f"Unparseable dates : {bad_dates}")
nav_raw = nav_raw.dropna(subset=["nav_date"])

# 1b · Sort
nav_raw = nav_raw.sort_values(["amfi_code", "nav_date"]).reset_index(drop=True)

# 1c · Deduplicate
before = len(nav_raw)
nav_raw = nav_raw.drop_duplicates(subset=["amfi_code", "nav_date"])
print(f"Duplicate rows removed : {before - len(nav_raw)}")

# 1d · Forward-fill missing business days per fund
filled_cells = 0
chunks = []
for code, grp in nav_raw.groupby("amfi_code"):
    grp = grp.set_index("nav_date").sort_index()
    full_idx  = pd.date_range(grp.index.min(), grp.index.max(), freq="B")
    grp       = grp.reindex(full_idx)
    filled_cells += int(grp["nav"].isna().sum())
    grp["nav"]       = grp["nav"].ffill()
    grp["amfi_code"] = code
    grp.index.name   = "nav_date"
    chunks.append(grp.reset_index())

nav = pd.concat(chunks, ignore_index=True)
print(f"Forward-filled gaps : {filled_cells:,} cells")

# 1e · Validate NAV > 0
bad_nav = (nav["nav"] <= 0).sum()
print(f"NAV ≤ 0 rows dropped : {bad_nav}")
nav = nav[nav["nav"] > 0].copy()

# 1f · Normalise date to ISO string
nav["nav_date"] = nav["nav_date"].dt.strftime("%Y-%m-%d")
nav = nav[["amfi_code", "nav_date", "nav"]]
nav.to_csv(PROC / "nav_history_clean.csv", index=False)

print(f"\n✅ nav_history_clean.csv written : {len(nav):,} rows")
display(nav.head(5))


In [ ]:
# Spot-check: NAV range per fund
nav_stats = (nav.groupby("amfi_code")["nav"]
               .agg(["min","max","count"])
               .merge(pd.read_csv(RAW/"fund_master.csv")[["amfi_code","scheme_name"]], on="amfi_code"))
display(nav_stats[["scheme_name","min","max","count"]].head(10))


---
## Task 2 · Clean `investor_transactions.csv`
Sources: `sip_transactions.csv` + `lumpsum_transactions.csv` + `investor_profiles.csv` (KYC / state)

**Rules applied**
| # | Rule | Detail |
|---|---|---|
| 1 | Unify column names | `sip_amount` → `amount`; add `transaction_type` column |
| 2 | Standardise `transaction_type` | Canonical map → `{SIP, Lumpsum, Redemption}` |
| 3 | Validate `amount > 0` | Drop null or non-positive amounts |
| 4 | Parse dates | `pd.to_datetime` → ISO-8601 string |
| 5 | KYC enum check | Must be `{Verified, Pending, Rejected}` |
| 6 | Join `kyc_status` + `state` | From `investor_profiles_clean.csv` |
| 7 | Deduplicate | Exact duplicate rows |


In [ ]:
sip  = pd.read_csv(RAW / "sip_transactions.csv")
lump = pd.read_csv(RAW / "lumpsum_transactions.csv")
inv  = pd.read_csv(RAW / "investor_profiles.csv")

print("SIP  :", sip.shape,  "| cols:", sip.columns.tolist())
print("Lump :", lump.shape, "| cols:", lump.columns.tolist())
print("Inv  :", inv.shape,  "| KYC:", inv.kyc_status.value_counts().to_dict())


In [ ]:
VALID_TXN  = {"SIP", "Lumpsum", "Redemption"}
VALID_KYC  = {"Verified", "Pending", "Rejected"}
TYPE_MAP   = {
    "sip":"SIP","Sip":"SIP","SIP":"SIP",
    "lumpsum":"Lumpsum","Lumpsum":"Lumpsum","LUMPSUM":"Lumpsum",
    "lump sum":"Lumpsum","lump_sum":"Lumpsum",
    "redemption":"Redemption","Redemption":"Redemption","redeem":"Redemption",
}

# Step 1 · Unify columns
sip  = sip.rename(columns={"sip_amount": "amount"})
sip["transaction_type"]  = "SIP"
lump["transaction_type"] = "Lumpsum"

COLS = ["investor_id","amfi_code","transaction_date","amount",
        "nav_at_purchase","units_allotted","transaction_type"]
for df in (sip, lump):
    for c in COLS:
        if c not in df.columns: df[c] = np.nan

txn = pd.concat([sip[COLS], lump[COLS]], ignore_index=True)
print(f"Combined rows : {len(txn):,}")

# Step 2 · Standardise transaction_type
txn["transaction_type"] = txn["transaction_type"].map(
    lambda x: TYPE_MAP.get(str(x).strip(), str(x).strip()))
non_std = ~txn["transaction_type"].isin(VALID_TXN)
if non_std.sum():
    print(f"⚠  {non_std.sum()} non-standard types → 'Unknown'")
    txn.loc[non_std, "transaction_type"] = "Unknown"
print("Type distribution:", txn["transaction_type"].value_counts().to_dict())

# Step 3 · Validate amount > 0
txn["amount"] = pd.to_numeric(txn["amount"], errors="coerce")
bad = txn["amount"].isna() | (txn["amount"] <= 0)
print(f"Bad amount rows dropped : {bad.sum()}")
txn = txn[~bad]

# Step 4 · Parse dates
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"], errors="coerce")
bad_d = txn["transaction_date"].isna().sum()
print(f"Unparseable dates dropped : {bad_d}")
txn = txn.dropna(subset=["transaction_date"])
txn["transaction_date"] = txn["transaction_date"].dt.strftime("%Y-%m-%d")
print(f"Date range : {txn['transaction_date'].min()} → {txn['transaction_date'].max()}")

# Step 5 · KYC enum check on investor_profiles
inv["kyc_status"] = inv["kyc_status"].astype(str).str.strip().str.title()
bad_kyc = ~inv["kyc_status"].isin(VALID_KYC)
if bad_kyc.sum():
    print(f"⚠  {bad_kyc.sum()} invalid KYC values relabelled 'Unknown'")
    inv.loc[bad_kyc, "kyc_status"] = "Unknown"
print("KYC distribution:", inv["kyc_status"].value_counts().to_dict())

# Step 6 · Derive state from city; join KYC + state
CITY_STATE = {
    "Mumbai":"Maharashtra","Pune":"Maharashtra","Delhi":"Delhi",
    "Bangalore":"Karnataka","Hyderabad":"Telangana","Chennai":"Tamil Nadu",
    "Kolkata":"West Bengal","Ahmedabad":"Gujarat",
}
inv["state"] = inv["city"].map(CITY_STATE).fillna("Unknown")
txn = txn.merge(inv[["investor_id","kyc_status","state"]], on="investor_id", how="left")
no_profile = txn["kyc_status"].isna().sum()
if no_profile:
    print(f"⚠  {no_profile} transactions with no investor profile")
    txn["kyc_status"] = txn["kyc_status"].fillna("No Profile")
    txn["state"]      = txn["state"].fillna("Unknown")

# Step 7 · Deduplicate
before = len(txn)
txn = txn.drop_duplicates()
print(f"Duplicate rows removed : {before - len(txn)}")

FINAL = ["investor_id","amfi_code","transaction_date","transaction_type",
         "amount","nav_at_purchase","units_allotted","kyc_status","state"]
txn = txn[FINAL].sort_values(["investor_id","transaction_date"]).reset_index(drop=True)
txn.to_csv(PROC / "investor_transactions_clean.csv", index=False)

print(f"\n✅ investor_transactions_clean.csv : {len(txn):,} rows × {txn.shape[1]} cols")
display(txn.head(5))


In [ ]:
# Summary statistics
print("Amount distribution:")
display(txn.groupby("transaction_type")["amount"].describe().round(2))
print("\nKYC × State cross-tab:")
display(pd.crosstab(txn["kyc_status"], txn["state"]))


---
## Task 3 · Clean `scheme_performance.csv`
Sources: `returns_summary.csv` + `fund_master.csv` (expense_ratio, metadata)

**Rules applied**
| # | Rule | Detail |
|---|---|---|
| 1 | Merge metadata | Join `fund_master` on `amfi_code` |
| 2 | Numeric validation | `pd.to_numeric(errors='coerce')` on all return columns |
| 3 | Flag anomalies | `\|return_1y\|>200`, `\|cagr_3y/5y\|>100`, `\|sharpe\|>5` |
| 4 | Expense ratio range | Must be `0.1% – 2.5%` (SEBI direct-plan cap) |
| 5 | Deduplicate | On `amfi_code` |


In [ ]:
ret = pd.read_csv(RAW / "returns_summary.csv")
fm  = pd.read_csv(RAW / "fund_master.csv")

print("returns_summary:", ret.shape, "| cols:", ret.columns.tolist())
print("fund_master    :", fm.shape,  "| cols:", fm.columns.tolist())
display(ret.head(3))


In [ ]:
RETURN_COLS    = ["return_1y_pct","return_3y_cagr_pct","return_5y_cagr_pct","sharpe_ratio"]
ANOMALY_RULES  = {"return_1y_pct":200,"return_3y_cagr_pct":100,"return_5y_cagr_pct":100,"sharpe_ratio":5}
EXPENSE_MIN, EXPENSE_MAX = 0.1, 2.5

# Step 1 · Merge metadata
perf = ret.merge(
    fm[["amfi_code","scheme_name","fund_house","category","sub_category",
        "risk_grade","expense_ratio_pct"]],
    on="amfi_code", how="left"
)
unmatched = perf["scheme_name"].isna().sum()
print(f"Unmatched amfi_codes : {unmatched}")

# Step 2 · Validate numeric
for col in RETURN_COLS:
    perf[col] = pd.to_numeric(perf[col], errors="coerce")
    print(f"  {col:<30} nulls={perf[col].isna().sum():>2}  range=[{perf[col].min():.2f}, {perf[col].max():.2f}]")

# Step 3 · Flag anomalies
perf["anomaly_flag"]   = False
perf["anomaly_reason"] = ""
for col, thresh in ANOMALY_RULES.items():
    mask = perf[col].abs() > thresh
    perf.loc[mask, "anomaly_flag"]    = True
    perf.loc[mask, "anomaly_reason"] += f"{col}>{thresh}; "
print(f"\nAnomalous rows flagged : {perf['anomaly_flag'].sum()}")
if perf["anomaly_flag"].sum():
    display(perf[perf["anomaly_flag"]][["amfi_code","scheme_name"]+RETURN_COLS])

# Step 4 · Expense ratio range check
perf["expense_ratio_pct"] = pd.to_numeric(perf["expense_ratio_pct"], errors="coerce")
perf["expense_ratio_ok"]  = perf["expense_ratio_pct"].between(EXPENSE_MIN, EXPENSE_MAX)
oor = (~perf["expense_ratio_ok"]).sum()
print(f"Out-of-range expense_ratio : {oor}  (valid range {EXPENSE_MIN}%–{EXPENSE_MAX}%)")
print(f"Expense ratio range in data: {perf['expense_ratio_pct'].min()}% – {perf['expense_ratio_pct'].max()}%")

# Step 5 · Deduplicate
before = len(perf)
perf = perf.drop_duplicates(subset=["amfi_code"])
print(f"Duplicate amfi_codes removed : {before - len(perf)}")

FINAL = ["amfi_code","scheme_name","fund_house","category","sub_category","risk_grade",
         "expense_ratio_pct","expense_ratio_ok",
         "return_1y_pct","return_3y_cagr_pct","return_5y_cagr_pct","sharpe_ratio",
         "anomaly_flag","anomaly_reason"]
perf = perf[FINAL].sort_values("amfi_code").reset_index(drop=True)
perf.to_csv(PROC / "scheme_performance_clean.csv", index=False)

print(f"\n✅ scheme_performance_clean.csv : {len(perf)} rows × {perf.shape[1]} cols")
display(perf[["scheme_name","risk_grade","expense_ratio_pct","return_1y_pct","sharpe_ratio"]])


---
## Tasks 4–10 · Clean Remaining 7 Source Files

Each file is cleaned with the universal rules (date parsing, numeric coercion, deduplication)
plus any file-specific logic noted in the table below.

| File | Specific Rule |
|---|---|
| `fund_master` | Parse `launch_date`; validate `expense_ratio_pct` and `aum_cr` |
| `investor_profiles` | KYC enum check; derive `state` from city |
| `risk_metrics` | Flag `beta < 0` as `beta_anomaly` (don't drop — retain for investigation) |
| `portfolio_holdings` | Re-normalise `weight_pct` to exactly 100% per fund; join `scheme_name` |
| `benchmark_nifty100` | Parse dates; validate `nifty100_tri > 0`; sort |
| `dividend_history` | Parse dates; validate `dividend_per_unit > 0`; sort |
| `sip_transactions` | Parse dates; validate `sip_amount > 0` (individual cleaned copy) |
| `lumpsum_transactions` | Parse dates; validate `amount > 0` (individual cleaned copy) |


In [ ]:
fm  = pd.read_csv(RAW / "fund_master.csv")
inv = pd.read_csv(RAW / "investor_profiles.csv")

# ── fund_master ──────────────────────────────────────────────────────────────
fm["launch_date"]        = pd.to_datetime(fm["launch_date"],       errors="coerce").dt.strftime("%Y-%m-%d")
fm["expense_ratio_pct"]  = pd.to_numeric( fm["expense_ratio_pct"], errors="coerce")
fm["aum_cr"]             = pd.to_numeric( fm["aum_cr"],            errors="coerce")
fm.to_csv(PROC / "fund_master_clean.csv", index=False)
print(f"✅ fund_master_clean.csv           : {len(fm)} rows")

# ── investor_profiles ────────────────────────────────────────────────────────
CITY_STATE = {"Mumbai":"Maharashtra","Pune":"Maharashtra","Delhi":"Delhi",
              "Bangalore":"Karnataka","Hyderabad":"Telangana","Chennai":"Tamil Nadu",
              "Kolkata":"West Bengal","Ahmedabad":"Gujarat"}
VALID_KYC  = {"Verified","Pending","Rejected"}
inv["registration_date"] = pd.to_datetime(inv["registration_date"], errors="coerce").dt.strftime("%Y-%m-%d")
inv["kyc_status"]        = inv["kyc_status"].astype(str).str.strip().str.title()
inv.loc[~inv["kyc_status"].isin(VALID_KYC), "kyc_status"] = "Unknown"
inv["state"]             = inv["city"].map(CITY_STATE).fillna("Unknown")
inv.to_csv(PROC / "investor_profiles_clean.csv", index=False)
print(f"✅ investor_profiles_clean.csv      : {len(inv)} rows  [state col added]")
print(f"   KYC: {inv['kyc_status'].value_counts().to_dict()}")

# ── risk_metrics ─────────────────────────────────────────────────────────────
rm = pd.read_csv(RAW / "risk_metrics.csv")
for c in rm.select_dtypes(exclude="object").columns:
    if c != "amfi_code":
        rm[c] = pd.to_numeric(rm[c], errors="coerce")
rm["beta_anomaly"] = rm["beta"] < 0
rm = rm.merge(fm[["amfi_code","scheme_name"]], on="amfi_code", how="left")
rm.to_csv(PROC / "risk_metrics_clean.csv", index=False)
print(f"✅ risk_metrics_clean.csv           : {len(rm)} rows  [beta_anomaly: {rm['beta_anomaly'].sum()} flagged]")

# ── portfolio_holdings ───────────────────────────────────────────────────────
ph = pd.read_csv(RAW / "portfolio_holdings.csv")
ph["weight_pct"]      = pd.to_numeric(ph["weight_pct"],      errors="coerce")
ph["market_value_cr"] = pd.to_numeric(ph["market_value_cr"], errors="coerce")
ph["weight_pct"]      = ph.groupby("amfi_code")["weight_pct"].transform(
    lambda g: (g / g.sum() * 100).round(4))
ph = ph.merge(fm[["amfi_code","scheme_name"]], on="amfi_code", how="left")
ph.to_csv(PROC / "portfolio_holdings_clean.csv", index=False)
print(f"✅ portfolio_holdings_clean.csv     : {len(ph)} rows  [weights re-normalised]")

# ── benchmark_nifty100 ───────────────────────────────────────────────────────
bm = pd.read_csv(RAW / "benchmark_nifty100.csv")
bm["date"]         = pd.to_datetime(bm["date"], errors="coerce").dt.strftime("%Y-%m-%d")
bm["nifty100_tri"] = pd.to_numeric(bm["nifty100_tri"], errors="coerce")
bm = bm.dropna().drop_duplicates(subset=["date"]).sort_values("date").reset_index(drop=True)
bm.to_csv(PROC / "benchmark_nifty100_clean.csv", index=False)
print(f"✅ benchmark_nifty100_clean.csv     : {len(bm):,} rows")

# ── dividend_history ─────────────────────────────────────────────────────────
dv = pd.read_csv(RAW / "dividend_history.csv")
dv["dividend_date"]     = pd.to_datetime(dv["dividend_date"], errors="coerce").dt.strftime("%Y-%m-%d")
dv["dividend_per_unit"] = pd.to_numeric(dv["dividend_per_unit"], errors="coerce")
dv = dv[dv["dividend_per_unit"] > 0].drop_duplicates()
dv = dv.sort_values(["amfi_code","dividend_date"]).reset_index(drop=True)
dv.to_csv(PROC / "dividend_history_clean.csv", index=False)
print(f"✅ dividend_history_clean.csv       : {len(dv)} rows")

# ── sip_transactions (individual copy) ───────────────────────────────────────
sip = pd.read_csv(RAW / "sip_transactions.csv")
sip["transaction_date"] = pd.to_datetime(sip["transaction_date"], errors="coerce").dt.strftime("%Y-%m-%d")
sip["sip_amount"]       = pd.to_numeric(sip["sip_amount"], errors="coerce")
sip = sip[sip["sip_amount"] > 0].reset_index(drop=True)
sip.to_csv(PROC / "sip_transactions_clean.csv", index=False)
print(f"✅ sip_transactions_clean.csv       : {len(sip):,} rows")

# ── lumpsum_transactions (individual copy) ───────────────────────────────────
lump = pd.read_csv(RAW / "lumpsum_transactions.csv")
lump["transaction_date"] = pd.to_datetime(lump["transaction_date"], errors="coerce").dt.strftime("%Y-%m-%d")
lump["amount"]           = pd.to_numeric(lump["amount"], errors="coerce")
lump = lump[lump["amount"] > 0].reset_index(drop=True)
lump.to_csv(PROC / "lumpsum_transactions_clean.csv", index=False)
print(f"✅ lumpsum_transactions_clean.csv   : {len(lump):,} rows")


In [ ]:
# Verify all 11 processed files exist
processed = sorted(PROC.glob("*.csv"))
print(f"Total processed CSVs : {len(processed)}")
summary = []
for p in processed:
    df = pd.read_csv(p)
    summary.append({"file": p.name, "rows": len(df), "cols": df.shape[1]})
display(pd.DataFrame(summary))


---
## Task 4 · SQLite Star Schema

**Schema design:** 2 dimensions + 4 fact tables

| Table | Type | Grain |
|---|---|---|
| `dim_fund` | Dimension | 1 row per AMFI scheme |
| `dim_date` | Dimension | 1 row per business date |
| `fact_nav` | Fact | 1 row per fund per day |
| `fact_transactions` | Fact | 1 row per transaction event |
| `fact_performance` | Fact | 1 row per fund per snapshot |
| `fact_aum` | Fact | 1 row per fund per date |


In [ ]:
SCHEMA_SQL = open(SQL / "schema.sql").read()
print(SCHEMA_SQL)


In [ ]:
# Apply schema to fresh database
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
conn.executescript(SCHEMA_SQL)
conn.commit()
conn.close()
print(f"✅ Schema applied to {DB_PATH.name}")


---
## Task 5 · Load Cleaned Datasets into SQLite
Using `SQLAlchemy create_engine` + `df.to_sql()`.  
Every table is verified: **source row count = database row count**.


In [ ]:
engine   = create_engine(f"sqlite:///{DB_PATH}", echo=False)
fm_clean = pd.read_csv(PROC / "fund_master_clean.csv")
nav_clean= pd.read_csv(PROC / "nav_history_clean.csv")
txn_clean= pd.read_csv(PROC / "investor_transactions_clean.csv")
perf_clean=pd.read_csv(PROC / "scheme_performance_clean.csv")
rm_clean  =pd.read_csv(PROC / "risk_metrics_clean.csv")

verification = []

def load_verify(table, df, src_label):
    df.to_sql(table, engine, if_exists="replace", index=False)
    with engine.connect() as c:
        db_n = c.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
    match = "✓ MATCH" if db_n == len(df) else f"✗ MISMATCH (src={len(df)}, db={db_n})"
    verification.append({"table": table, "source_rows": len(df), "db_rows": db_n, "status": match})
    print(f"  {table:<24}  src={len(df):>6,}  db={db_n:>6,}  {match}  ← {src_label}")


In [ ]:
print("Loading dim_fund …")
dim_fund = fm_clean[["amfi_code","scheme_name","fund_house","category","sub_category",
                      "risk_grade","benchmark","launch_date","aum_cr","expense_ratio_pct"]]
load_verify("dim_fund", dim_fund, "fund_master_clean.csv")

print("Loading fact_nav …")
load_verify("fact_nav", nav_clean[["amfi_code","nav_date","nav"]], "nav_history_clean.csv")

print("Building dim_date from nav dates …")
dates_raw  = pd.to_datetime(nav_clean["nav_date"].unique())
dim_date   = pd.DataFrame({
    "full_date":   dates_raw.strftime("%Y-%m-%d"),
    "year":        dates_raw.year,
    "quarter":     dates_raw.quarter,
    "month":       dates_raw.month,
    "month_name":  dates_raw.strftime("%B"),
    "week_num":    dates_raw.isocalendar().week.values,
    "day":         dates_raw.day,
    "day_of_week": dates_raw.strftime("%A"),
    "is_weekend":  (dates_raw.dayofweek >= 5).astype(int),
}).sort_values("full_date").reset_index(drop=True)
load_verify("dim_date", dim_date, "derived from nav_date")

print("Loading fact_transactions …")
txn_cols = ["investor_id","amfi_code","transaction_date","transaction_type",
            "amount","nav_at_purchase","units_allotted","kyc_status","state"]
load_verify("fact_transactions", txn_clean[txn_cols], "investor_transactions_clean.csv")

print("Loading fact_performance …")
perf_db = perf_clean.merge(
    rm_clean[["amfi_code","annualised_volatility_pct","var_95_daily_pct",
              "max_drawdown_pct","beta"]], on="amfi_code", how="left")
perf_db["as_of_date"]   = "2024-03-31"
perf_db["anomaly_flag"] = perf_db["anomaly_flag"].astype(int)
perf_cols = ["amfi_code","as_of_date","return_1y_pct","return_3y_cagr_pct",
             "return_5y_cagr_pct","sharpe_ratio","var_95_daily_pct","max_drawdown_pct",
             "annualised_volatility_pct","beta","expense_ratio_pct",
             "anomaly_flag","anomaly_reason"]
perf_db = perf_db[perf_cols].rename(columns={
    "var_95_daily_pct":         "var_95_pct",
    "annualised_volatility_pct":"annualised_vol_pct"})
load_verify("fact_performance", perf_db, "scheme_performance_clean.csv + risk_metrics_clean.csv")

print("Loading fact_aum …")
aum_db = fm_clean[["amfi_code","aum_cr"]].copy()
aum_db["as_of_date"] = "2024-03-31"
load_verify("fact_aum", aum_db, "fund_master_clean.csv (aum_cr)")

engine.dispose()


In [ ]:
# Row-count verification summary
print("\n" + "="*60)
print("  ROW COUNT VERIFICATION SUMMARY")
print("="*60)
display(pd.DataFrame(verification))
print(f"\nDatabase file : {DB_PATH}")
print(f"File size     : {DB_PATH.stat().st_size / 1024:.0f} KB")


---
## Task 6 · 10 Analytical SQL Queries

All queries run against `bluestock_mf.db` using `pd.read_sql`.


### Q01 — Top 5 Funds by AUM

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
df = pd.read_sql("""
SELECT  f.scheme_name, f.category, f.sub_category,
        a.aum_cr, RANK() OVER (ORDER BY a.aum_cr DESC) AS aum_rank
FROM    fact_aum a JOIN dim_fund f ON f.amfi_code = a.amfi_code
ORDER BY a.aum_cr DESC LIMIT 5;
""", engine)
engine.dispose()
display(df)


### Q02 — Average NAV per Month (last 12 months, all funds)

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
df = pd.read_sql("""
SELECT  d.year, d.month, d.month_name,
        ROUND(AVG(n.nav), 2) AS avg_nav,
        COUNT(DISTINCT n.amfi_code) AS funds_included
FROM    fact_nav n JOIN dim_date d ON d.full_date = n.nav_date
WHERE   d.full_date >= DATE('2024-03-31','-12 months')
GROUP BY d.year, d.month ORDER BY d.year, d.month;
""", engine)
engine.dispose()
display(df)


### Q03 — SIP Year-over-Year Growth

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
df = pd.read_sql("""
SELECT  SUBSTR(transaction_date,1,4) AS year,
        COUNT(*) AS sip_count,
        ROUND(SUM(amount),0) AS total_invested,
        ROUND(AVG(amount),2) AS avg_ticket,
        COUNT(DISTINCT investor_id) AS unique_investors
FROM    fact_transactions WHERE transaction_type = 'SIP'
GROUP BY year ORDER BY year;
""", engine)
engine.dispose()
display(df)


### Q04 — Transactions by State

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
df = pd.read_sql("""
SELECT  state, COUNT(*) AS txn_count,
        ROUND(SUM(amount)/1e6,2) AS total_invested_mn,
        COUNT(DISTINCT investor_id) AS investors,
        ROUND(AVG(amount),0) AS avg_ticket
FROM    fact_transactions WHERE state IS NOT NULL
GROUP BY state ORDER BY total_invested_mn DESC;
""", engine)
engine.dispose()
display(df)


### Q05 — Funds with Expense Ratio < 1%

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
df = pd.read_sql("""
SELECT  f.scheme_name, f.category, f.risk_grade,
        p.expense_ratio_pct, p.sharpe_ratio, p.return_1y_pct
FROM    fact_performance p JOIN dim_fund f ON f.amfi_code = p.amfi_code
WHERE   p.expense_ratio_pct < 1.0
ORDER BY p.expense_ratio_pct ASC;
""", engine)
engine.dispose()
display(df)


### Q06 — Risk-Return Summary by Risk Grade

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
df = pd.read_sql("""
SELECT  f.risk_grade, COUNT(*) AS funds,
        ROUND(AVG(p.return_1y_pct),2)      AS avg_1y_return,
        ROUND(AVG(p.sharpe_ratio),4)        AS avg_sharpe,
        ROUND(AVG(p.max_drawdown_pct),2)    AS avg_max_drawdown,
        ROUND(AVG(p.annualised_vol_pct),2)  AS avg_volatility
FROM    fact_performance p JOIN dim_fund f ON f.amfi_code = p.amfi_code
GROUP BY f.risk_grade ORDER BY avg_sharpe DESC;
""", engine)
engine.dispose()
display(df)


### Q07 — SIP Volume by Quarter

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
df = pd.read_sql("""
SELECT  d.year, d.quarter,
        ROUND(SUM(t.amount)/1e6,2)      AS sip_volume_mn,
        COUNT(*)                         AS sip_count,
        COUNT(DISTINCT t.investor_id)    AS active_investors
FROM    fact_transactions t JOIN dim_date d ON d.full_date = t.transaction_date
WHERE   t.transaction_type = 'SIP'
GROUP BY d.year, d.quarter ORDER BY d.year, d.quarter;
""", engine)
engine.dispose()
display(df)


### Q08 — Investor Cohort Analysis (first SIP year)

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
df = pd.read_sql("""
WITH first_txn AS (
    SELECT investor_id, MIN(SUBSTR(transaction_date,1,4)) AS cohort_year
    FROM   fact_transactions WHERE transaction_type='SIP'
    GROUP BY investor_id
)
SELECT  f.cohort_year,
        COUNT(DISTINCT t.investor_id)  AS investors,
        COUNT(*)                        AS total_sips,
        ROUND(SUM(t.amount),0)          AS total_invested,
        ROUND(AVG(t.amount),2)          AS avg_sip_amount
FROM    fact_transactions t JOIN first_txn f ON f.investor_id = t.investor_id
WHERE   t.transaction_type = 'SIP'
GROUP BY f.cohort_year ORDER BY f.cohort_year;
""", engine)
engine.dispose()
display(df)


### Q09 — Fund Performance Leaderboard (1-Year, all funds)

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
df = pd.read_sql("""
SELECT  f.scheme_name, f.category, f.risk_grade,
        p.return_1y_pct, p.return_3y_cagr_pct, p.sharpe_ratio,
        CASE WHEN p.return_1y_pct >= 0 THEN 'Positive' ELSE 'Negative' END AS direction
FROM    fact_performance p JOIN dim_fund f ON f.amfi_code = p.amfi_code
ORDER BY p.return_1y_pct DESC;
""", engine)
engine.dispose()
display(df)


### Q10 — NAV 5-Year Total Return (first vs last NAV per fund)

In [ ]:
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
df = pd.read_sql("""
WITH first_nav AS (
    SELECT amfi_code, nav AS nav_start, nav_date AS start_date
    FROM   fact_nav WHERE nav_date=(SELECT MIN(nav_date) FROM fact_nav)
),
last_nav AS (
    SELECT amfi_code, nav AS nav_end, nav_date AS end_date
    FROM   fact_nav WHERE nav_date=(SELECT MAX(nav_date) FROM fact_nav)
)
SELECT  f.scheme_name, f.category,
        fn.nav_start, ln.nav_end,
        fn.start_date, ln.end_date,
        ROUND((ln.nav_end-fn.nav_start)/fn.nav_start*100,2) AS total_return_pct
FROM    first_nav fn
JOIN    last_nav  ln ON ln.amfi_code = fn.amfi_code
JOIN    dim_fund   f ON  f.amfi_code = fn.amfi_code
ORDER BY total_return_pct DESC;
""", engine)
engine.dispose()
display(df)


---
## Task 7 · Data Dictionary Preview

The full data dictionary is in `reports/data_dictionary.md`.  
Below is an inline column-level reference for the six SQLite tables.


In [ ]:
dd_path = BASE / "reports" / "data_dictionary.md"
if dd_path.exists():
    from IPython.display import Markdown
    display(Markdown(dd_path.read_text()))
else:
    print("data_dictionary.md not found — run pipeline_day2.py first")


---
## Task 8 · Git Commit

Commits all cleaned CSVs, the SQLite DB, SQL files, and reports.


In [ ]:
import subprocess

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=BASE)
    out = (r.stdout + r.stderr).strip()
    print(out if out else f"[no output] returncode={r.returncode}")

run(["git", "config", "user.email", "capstone@bluestock.in"])
run(["git", "config", "user.name",  "Bluestock Capstone"])
run(["git", "-C", str(BASE), "add",
     "data/processed/", "data/db/", "sql/", "reports/", "pipeline_day2.py"])
run(["git", "-C", str(BASE), "commit", "-m", "Day 2: Cleaned data + SQLite DB loaded"])
run(["git", "-C", str(BASE), "log", "--oneline", "-5"])


---
## Pipeline Complete ✅

| Deliverable | Location |
|---|---|
| 11 cleaned CSVs | `data/processed/` |
| SQLite database | `data/db/bluestock_mf.db` |
| Star schema DDL | `sql/schema.sql` |
| 10 SQL queries | `sql/queries.sql` |
| Data dictionary | `reports/data_dictionary.md` |
| Pipeline script | `pipeline_day2.py` |

---
*Bluestock MF Capstone · Day 2 · Data Cleaning & SQLite Load*
